# افزودن مقاله‌های تازه به مجموعه و سایت (نسخه‌ی 13)

PDFها باید در پوشه‌ی `proDownloads` در Drive باشند، با همان نام‌هایی که دانلود شده‌اند (مثل `petrov2015.pdf`،
`kato2018.pdf`، `pack2004.pdf`؛ فهرست کامل در کد). مقاله‌هایی که از قبل اضافه شده‌اند دوباره اضافه نمی‌شوند. فقط یک سلول دارد: روی ▶️ بزنید و اجازه‌ی دسترسی به
Google Drive را بدهید. کارت گرافیک لازم نیست. حدود ۳۰ تا ۶۰ دقیقه طول می‌کشد.

۱) هر PDF با نام DOI در پوشه‌ی دسته و بلور خودش در `RAG-Optics` **کپی** می‌شود (چیزی پاک یا جابه‌جا نمی‌شود)؛
مقاله‌هایی که در فهرست اصلی نیستند به آن اضافه می‌شوند (یک نسخه‌ی پشتیبان از فهرست کنار آن ذخیره می‌شود).
۲) مقاله‌ها پردازش و وارد پایگاه می‌شوند. ۳) PDFها و پایگاه روی سایت به‌روز می‌شوند.
اگر قطع شد دوباره اجرا کنید؛ کارهای انجام‌شده تکرار نمی‌شوند.

In [ ]:
#@title ▶️ افزودن مقاله‌ها
DOWNLOADS = '/content/drive/MyDrive/proDownloads'  #@param {type:"string"}

import os
# Keep Colab's preinstalled JAX/TensorFlow from reserving the GPU memory
# (otherwise the embedding model and Ollama run out of memory).
os.environ.update({'JAX_PLATFORMS': 'cpu', 'XLA_PYTHON_CLIENT_PREALLOCATE': 'false',
                   'TF_FORCE_GPU_ALLOW_GROWTH': 'true', 'USE_TF': '0', 'USE_FLAX': '0',
                   'TF_CPP_MIN_LOG_LEVEL': '3'})
from google.colab import drive
drive.mount('/content/drive')

WORK = '/content/drive/MyDrive/matrag_data'
os.environ['MATRAG_DATA_DIR'] = f'{WORK}/data'
os.environ['MATRAG_STORAGE_DIR'] = '/content/storage'
os.environ['MATRAG_STORAGE_BACKUP_DIR'] = f'{WORK}/storage'
os.environ['MATRAG_CORPUS'] = 'rag-optics'
print('Copying the database from Drive...')
!mkdir -p /content/storage && cp -rT "{WORK}/storage" /content/storage

BRANCH = "claude/festive-bardeen-nnw123"  #@param {type:"string"}
# Leave the project folder before deleting it (re-running this cell used to fail
# with "Unable to read current working directory").
%cd /content
!rm -rf /content/RAG && git clone -q --depth 1 -b {BRANCH} https://github.com/aminsadidi/RAG.git /content/RAG
%cd /content/RAG
!pip install -q -e . 2>&1 | grep -iE "^error" || true
# An editable install is only seen by Python after a restart; adding the source
# folder to the path makes it importable in this session right away.
import sys
if '/content/RAG/src' not in sys.path:
    sys.path.insert(0, '/content/RAG/src')

# File in the downloads folder -> DOI, category, material; rows only for papers the master index lacks.
PAPERS = {
    'pack2003.pdf': ('10.1364/josab.20.002109', '03_Niobates_Tantalates_Iodates', 'KNbO3', {
        'title': 'Measurement of the χ^(2) tensor of the potassium niobate crystal',
        'authors': 'Michael V. Pack; Darrell J. Armstrong; Arlee V. Smith', 'year': 2003,
        'journal': 'Journal of the Optical Society of America B'}),
    'pack2004.pdf': ('10.1364/ao.43.003319', '02_Phosphates_Arsenates_KDP', 'RTP_RbTiOPO4', None),
    'pack2005.pdf': ('10.1364/josab.22.000417', '01_Borates', 'YCOB_GdCOB_ReCOB', {
        'title': 'Measurement of the χ^(2) tensors of GdCa4O(BO3)3 and YCa4O(BO3)3 crystals',
        'authors': 'Michael V. Pack; Darrell J. Armstrong; Arlee V. Smith; Gerard Aka; Bernard Ferrand; Denis Pelenc',
        'year': 2005, 'journal': 'Journal of the Optical Society of America B'}),
    'hellwig1998.pdf': ('10.1016/s0038-1098(98)00538-9', '01_Borates', 'BiBO_BiB3O6', None),
    'jerphagnon1970.pdf': ('10.1103/physrevb.1.1739', '02_Phosphates_Arsenates_KDP', 'ADP_NH4H2PO4', None),
    'li2016.pdf': ('10.1016/j.optmat.2016.10.023', '01_Borates', 'LCB_La2CaB10O19', None),
    # 2026-10-07: sources of the NLO data (docs/nlo_sources.md) and SPDC theory
    'ljunggren2005.pdf': ('10.1103/physreva.72.062301', '08_Theory_Dispersion_Models', 'Phase_matching_theory_birefringence', {'title': 'Optimal focusing for maximal collection of entangled narrow-band photon pairs into single-mode fibers', 'authors': 'Daniel Ljunggren; Maria Tengner', 'year': 2005, 'journal': 'Physical Review A'}),
    'tzankov2005.pdf': ('10.1364/ao.44.006971', '08_Theory_Dispersion_Models', 'Nonlinear_coefficients_Miller_rule', {'title': 'Effective second-order nonlinearity in acentric optical crystals with low symmetry', 'authors': 'Pancho Tzankov; Valentin Petrov', 'year': 2005, 'journal': 'Applied Optics'}),
    'kato1994.pdf': ('10.1109/3.362711', '01_Borates', 'LBO_LiB3O5', {'title': 'Temperature-tuned 90° phase-matching properties of LiB3O5', 'authors': 'K. Kato', 'year': 1994, 'journal': 'IEEE Journal of Quantum Electronics'}),
    'fedrizzi2007.pdf': ('10.1364/oe.15.015377', '02_Phosphates_Arsenates_KDP', 'KTP_KTiOPO4', {'title': 'A wavelength-tunable fiber-coupled source of narrowband entangled photons', 'authors': 'Alessandro Fedrizzi; Thomas Herbst; Andreas Poppe; Thomas Jennewein; Anton Zeilinger', 'year': 2007, 'journal': 'Optics Express'}),
    'gayer2010.pdf': ('10.1007/s00340-010-4203-7', '03_Niobates_Tantalates_Iodates', 'LiNbO3_LN_PPLN', {'title': 'Erratum to: Temperature and wavelength dependent refractive index equations for MgO-doped congruent and stoichiometric LiNbO3', 'authors': 'O. Gayer; Z. Sacks; E. Galun; A. Arie', 'year': 2010, 'journal': 'Applied Physics B'}),
    'dolev2009.pdf': ('10.1007/s00340-009-3502-3', '03_Niobates_Tantalates_Iodates', 'LiTaO3_LT_PPLT', {'title': 'Linear and nonlinear optical properties of MgO:LiTaO3', 'authors': 'I. Dolev; A. Ganany-Padowicz; O. Gayer; A. Arie; J. Mangin; G. Gadret', 'year': 2009, 'journal': 'Applied Physics B'}),
    'petrov2015.pdf': ('10.1016/j.pquantelec.2015.04.001', '08_Theory_Dispersion_Models', 'Reviews_NLO_crystals_databases', {'title': 'Frequency down-conversion of solid-state laser sources to the mid-infrared spectral range using non-oxide nonlinear crystals', 'authors': 'Valentin Petrov', 'year': 2015, 'journal': 'Progress in Quantum Electronics'}),
    'petrov2012.pdf': ('10.1016/j.optmat.2011.03.042', '08_Theory_Dispersion_Models', 'Reviews_NLO_crystals_databases', {'title': 'Parametric down-conversion devices: The coverage of the mid-infrared spectral range by solid-state laser sources', 'authors': 'Valentin Petrov', 'year': 2012, 'journal': 'Optical Materials'}),
    'grice1997.pdf': ('10.1103/physreva.56.1627', '08_Theory_Dispersion_Models', 'Phase_matching_theory_birefringence', {'title': 'Spectral information and distinguishability in type-II down-conversion with a broadband pump', 'authors': 'W. P. Grice; I. A. Walmsley', 'year': 1997, 'journal': 'Physical Review A'}),
    'zhai2013.pdf': ('10.1016/j.optmat.2013.09.017', '01_Borates', 'KBBF_KBe2BO3F2', {'title': 'Measurement of thermal refractive index coefficients of nonlinear optical crystal RbBe2BO3F2', 'authors': 'Naixia Zhai; Lirong Wang; Lijuan Liu; Xiaoyang Wang; Yong Zhu; Chuangtian Chen', 'year': 2013, 'journal': 'Optical Materials'}),
    'kato2018.pdf': ('10.1088/1555-6611/aac9df', '01_Borates', 'LBO_LiB3O5', {'title': 'New thermo-optic dispersion formula for LiB3O5', 'authors': 'K Kato; S G Grechin; N Umemura', 'year': 2018, 'journal': 'Laser Physics'}),
    'komatsu1997.pdf': ('10.1063/1.119210', '01_Borates', 'SBBO_KABO_other_borates', {'title': 'Growth and ultraviolet application of Li2B4O7 crystals: Generation of the fourth and fifth harmonics of Nd:Y3Al5O12 lasers', 'authors': 'R. Komatsu; T. Sugawara; K. Sassa; N. Sarukura; Z. Liu; S. Izumida; Y. Segawa; S. Uda; T. Fukuda; K. Yamanouchi', 'year': 1997, 'journal': 'Applied Physics Letters'}),
    'bennink2010.pdf': ('10.1103/physreva.81.053805', '08_Theory_Dispersion_Models', 'Phase_matching_theory_birefringence', {'title': 'Optimal collinear Gaussian beams for spontaneous parametric down-conversion', 'authors': 'Ryan S. Bennink', 'year': 2010, 'journal': 'Physical Review A'}),
    'hong1985.pdf': ('10.1103/physreva.31.2409', '08_Theory_Dispersion_Models', 'Phase_matching_theory_birefringence', {'title': 'Theory of parametric frequency down conversion of light', 'authors': 'C. K. Hong; L. Mandel', 'year': 1985, 'journal': 'Physical Review A'}),
    'sugawara1998.pdf': ('10.1016/s0038-1098(98)00190-2', '01_Borates', 'SBBO_KABO_other_borates', {'title': 'Linear and nonlinear optical properties of lithium tetraborate', 'authors': 'Tamotsu Sugawara; Ryuichi Komatsu; Satoshi Uda', 'year': 1998, 'journal': 'Solid State Communications'}),
    'ghosh1992.pdf': ('10.1117/12.637003', '02_Phosphates_Arsenates_KDP', 'KDP_DKDP_KH2PO4', {'title': 'Dispersion of thermo-optic coefficients and temperature-dependent nonlinear optical devices of some nonlinear crystals', 'authors': 'Gorachand Ghosh', 'year': 1992, 'journal': 'SPIE Proceedings'}),
    'evans2010.pdf': ('10.1103/physrevlett.105.253601', '02_Phosphates_Arsenates_KDP', 'KTP_KTiOPO4', {'title': 'Bright Source of Spectrally Uncorrelated Polarization-Entangled Photons with Nearly Single-Mode Emission', 'authors': 'P. G. Evans; R. S. Bennink; W. P. Grice; T. S. Humble; J. Schaake', 'year': 2010, 'journal': 'Physical Review Letters'}),
    'umemura2001.pdf': ('10.1364/assl.1999.pd15', '01_Borates', 'CLBO_CsLiB6O10', {'title': 'New data on the phase-matching properties of CsLiB6O10', 'authors': 'N. Umemura; K. Yoshida; T. Kamimura; Y. Mori; T. Sasaki; K. Kato', 'year': 2001, 'journal': 'Advanced Solid State Lasers'}),
    'mosley2008.pdf': ('10.1103/physrevlett.100.133601', '02_Phosphates_Arsenates_KDP', 'KDP_DKDP_KH2PO4', {'title': 'Heralded Generation of Ultrafast Single Photons in Pure Quantum States', 'authors': 'Peter J. Mosley; Jeff S. Lundeen; Brian J. Smith; Piotr Wasylczyk; Alfred B. U’Ren; Christine Silberhorn; Ian A. Walmsley', 'year': 2008, 'journal': 'Physical Review Letters'}),
    'petrov1998.pdf': ('10.1063/1.368904', '01_Borates', 'SBBO_KABO_other_borates', {'title': 'Vacuum ultraviolet application of Li2B4O7 crystals: Generation of 100 fs pulses down to 170 nm', 'authors': 'V. Petrov; F. Rotermund; F. Noack; R. Komatsu; T. Sugawara; S. Uda', 'year': 1998, 'journal': 'Journal of Applied Physics'}),
    'ghotbi2004.pdf': ('10.1364/opex.12.006002', '01_Borates', 'BiBO_BiB3O6', None),
    'law2000.pdf': ('10.1103/physrevlett.84.5304', '08_Theory_Dispersion_Models', 'Phase_matching_theory_birefringence', {'title': 'Continuous Frequency Entanglement: Effective Finite Hilbert Space and Entropy Control', 'authors': 'C. K. Law; I. A. Walmsley; J. H. Eberly', 'year': 2000, 'journal': 'Physical Review Letters'}),
    'miyata2009.pdf': ('10.1364/ol.34.000500', '01_Borates', 'BiBO_BiB3O6', {'title': 'Phase-matched pure χ^(3) third-harmonic generation in noncentrosymmetric BiB3O6', 'authors': 'Kentaro Miyata; Nobuhiro Umemura; Kiyoshi Kato', 'year': 2009, 'journal': 'Optics Letters'}),
    'shoji1999.pdf': ('10.1364/josab.16.000620', '01_Borates', 'BBO_beta-BaB2O4', {'title': 'Absolute measurement of second-order nonlinear-optical coefficients of β-BaB2O4 for visible to ultraviolet second-harmonic wavelengths', 'authors': 'Ichiro Shoji; Hirotaka Nakamura; Keisuke Ohdaira; Takashi Kondo; Ryoichi Ito; Tsutomu Okamoto; Koichi Tatsuki; Shigeo Kubota', 'year': 1999, 'journal': 'Journal of the Optical Society of America B'}),
    'hong1987.pdf': ('10.1103/physrevlett.59.2044', '08_Theory_Dispersion_Models', 'Phase_matching_theory_birefringence', {'title': 'Measurement of subpicosecond time intervals between two photons by interference', 'authors': 'C. K. Hong; Z. Y. Ou; L. Mandel', 'year': 1987, 'journal': 'Physical Review Letters'}),
    # 2026-10-10: nonlinear coefficients of GaN, ZnTe and BaGa4Se7 (nonlinear_coefficients.yml)
    'wagner1998.pdf': ('10.1103/physrevb.58.10494', '05_Semiconductors_QPM', 'ZnSe_ZnTe', None),
    'sanford2005.pdf': ('10.1063/1.1852695', '05_Semiconductors_QPM', 'GaN_AlN', {'title': 'Measurement of second order susceptibilities of GaN and AlGaN', 'authors': 'N. A. Sanford; A. V. Davydov; D. V. Tsvetkov; A. V. Dmitriev; S. Keller; U. K. Mishra; S. P. DenBaars; S. S. Park; J. Y. Han; R. J. Molnar', 'year': 2005, 'journal': 'Journal of Applied Physics'}),
    'zhang2015.pdf': ('10.1364/oe.23.000552', '04_Mid-IR_Chalcopyrites_Chalcogenides', 'BaGa4Se7_BaGa4S7', None),
    # 2026-10-10: sources of the thermo-optic data (thermo_optic.yml) and the two reference books
    'tanaka1998.pdf': ('10.1364/ao.37.000561', '04_Mid-IR_Chalcopyrites_Chalcogenides', 'AgGaSe2', {'title': 'Thermo-optic dispersion formula of AgGaSe2 and its practical applications', 'authors': 'Eiko Tanaka; Kiyoshi Kato', 'year': 1998, 'journal': 'Applied Optics'}),
    'mikami2011.pdf': ('10.1063/1.3525800', '08_Theory_Dispersion_Models', 'Sellmeier_equation_fitting', None),
    'wei2018.pdf': ('10.1364/ome.8.000485', '05_Semiconductors_QPM', 'GaP_OP-GaP', None),
    'loiko2015.pdf': ('10.1364/ome.5.001089', '01_Borates', 'YCOB_GdCOB_ReCOB', None),
    'kato2019.pdf': ('10.3390/cryst9030129', '02_Phosphates_Arsenates_KDP', 'KTP_KTiOPO4', None),  # AgGaS2; the master index files it here
    # the books: Dmitriev's 39 MB scan is converted with OCR (ocr_papers.txt) and is over Cloudflare's 25 MB per
    # file, so it is searchable but has no page view on the site
    'dmitriev1999.pdf': ('10.1007/978-3-540-46793-9', '08_Theory_Dispersion_Models', 'Reviews_NLO_crystals_databases', {'title': 'Handbook of Nonlinear Optical Crystals (3rd ed.)', 'authors': 'Valentin G. Dmitriev; Gagik G. Gurzadyan; David N. Nikogosyan', 'year': 1999, 'journal': 'Springer Series in Optical Sciences 64 (book)'}),
    'nonlinear-optical-crystals-a-complete-survey-2005.pdf': ('10.1007/b138685', '08_Theory_Dispersion_Models', 'Reviews_NLO_crystals_databases', {'title': 'Nonlinear Optical Crystals: A Complete Survey', 'authors': 'David N. Nikogosyan', 'year': 2005, 'journal': 'Springer (book)'}),

}
import csv, json, pathlib, shutil
from matrag.catalog import PAPERS_DIR, doi_key, index_file, load_catalog, plan
root = pathlib.Path(open(f'{WORK}/data/rag-optics/corpus_root.txt').read().strip())
downloads = pathlib.Path(DOWNLOADS)
known = {e.key for e in load_catalog(root)}
print('1/4 Looking for the papers in the collection...')
have = {i.doc_id: i.pdf for i in plan(root).items if i.pdf is not None}  # a PDF under any file name
rows, placed = [], 0
for name, (doi, category, material, meta) in PAPERS.items():
    pdf = downloads / name
    target = root / PAPERS_DIR / category / material / (doi_key(doi) + '.pdf')
    if target.exists() or doi_key(doi) in have:
        print('already in the collection:', name, '->', (target if target.exists() else have[doi_key(doi)]).relative_to(root))
    elif not pdf.exists():
        print('⚠️ not found, skipped:', pdf)
        continue
    else:
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(pdf, target)  # a copy: the downloads folder stays as it is
        placed += 1
        print('added:', name, '->', target.relative_to(root))
    if doi_key(doi) not in known and meta:
        rows.append({'category': category, 'material': material, 'doi': doi, 'doi_link': 'https://doi.org/' + doi,
                     'status': 'downloaded', **meta})
if rows:
    path = index_file(root)
    shutil.copy2(path, str(path) + '.before_new_papers')  # backup of the master index
    if path.suffix == '.csv':
        with path.open(encoding='utf-8-sig', newline='') as f:
            fields = csv.DictReader(f).fieldnames
        with path.open('a', encoding='utf-8', newline='') as f:
            csv.DictWriter(f, fieldnames=fields, extrasaction='ignore').writerows(rows)
    else:
        with path.open('a', encoding='utf-8') as f:
            f.writelines(json.dumps(r, ensure_ascii=False) + '\n' for r in rows)
    print(len(rows), 'rows added to', path.name)
print(f'✅ {placed} PDFs copied')

# Keys: matrag_data/qdrant.env (Qdrant) and matrag_data/cloudflare.env (the site's PDFs) on Drive,
# as KEY=value lines; the Qdrant ones can also come from Colab Secrets.
import os
KEYS_FILE = f'{WORK}/qdrant.env'
for keys_file in (KEYS_FILE, f'{WORK}/cloudflare.env'):
    if os.path.exists(keys_file):
        for line in open(keys_file, encoding='utf-8'):
            key, sep, value = line.strip().partition('=')
            if sep and not key.startswith('#'):
                os.environ[key.strip()] = value.strip().strip('"')
if not os.environ.get('QDRANT_API_KEY'):
    from google.colab import userdata
    for key in ('QDRANT_URL', 'QDRANT_API_KEY'):
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            pass
assert os.environ.get('QDRANT_URL') and os.environ.get('QDRANT_API_KEY'), f'Qdrant key not found ({KEYS_FILE})'

%cd {WORK}
# 2. Only the new papers are converted and embedded (the rest is already in the database)
print('2/4 Indexing the new papers (the rest are skipped as unchanged)...')
!matrag --corpus rag-optics ingest 2>&1 | grep -vE "Warning|Loading|it/s\]" | tee -a "{WORK}/logs/ingest.log"
# 3. The site: PDFs for the page view, then the chunks for the search
if os.environ.get('CLOUDFLARE_API_TOKEN'):
    print('3/4 Copying all PDFs from Drive for the site: 10-20 minutes with no output, it is not stuck...')
    !matrag --corpus rag-optics export-pdfs /content/RAG/web-pdfs/pdfs 2>&1 | grep -vE "Warning|Loading"
    if not os.path.exists('/content/node/bin/node'):
        !curl -fsSL https://nodejs.org/dist/v22.12.0/node-v22.12.0-linux-x64.tar.xz | tar -xJ -C /content && mv /content/node-v22.12.0-linux-x64 /content/node
    os.environ['PATH'] = '/content/node/bin:' + os.environ['PATH']
    print('   Uploading the PDFs to the site (a few minutes)...')
    !cd /content/RAG/web-pdfs && npx --yes wrangler@4 deploy 2>&1 | grep -vE "telemetry|^$" | tail -4
else:
    print('ℹ️ No matrag_data/cloudflare.env: PDFs not uploaded (the page view needs them)')
print('4/4 Sending the new chunks to the site search (Qdrant)...')
!matrag --corpus rag-optics export-qdrant 2>&1 | grep -vE "Warning|Loading"
%cd /content/RAG
!matrag --corpus rag-optics status 2>&1 | grep -vE "Warning"
# 5. The handled files leave the downloads folder for proDownloads/processed (moved, never deleted), so the
# folder only holds what has not been added yet. A file that was not found or failed stays where it is.
done_dir = downloads / 'processed'
done_dir.mkdir(exist_ok=True)
in_collection = {i.doc_id for i in plan(root).items if i.pdf is not None}
moved = 0
for name, (doi, *_rest) in PAPERS.items():
    pdf = downloads / name
    if pdf.exists() and doi_key(doi) in in_collection:
        shutil.move(str(pdf), str(done_dir / name)); moved += 1
print(f'{moved} handled PDFs moved to {done_dir}')
print('✅ Done: https://matrag.ethanjamescarter1995.workers.dev')